In [2]:
# import sys
# !{sys.executable} -m pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [ ]:
import spacy
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import nltk
from nltk.corpus import stopwords

import en_core_web_sm

import pyLDAvis
import pyLDAvis.lda_model 

from src.config import PARQUET_FILE

In [ ]:
# Carga de datos
df = pd.read_parquet(PARQUET_FILE)

In [ ]:
# Cargamos el modelo de lenguaje (el parámetro 'disable' acelera el proceso)
nlp = en_core_web_sm.load(disable=['ner', 'parser'])

print("Modelo cargado correctamente!")

def lemmatize_text(text):
    # Procesamos el texto
    doc = nlp(text.lower())
    
    # Extraemos el lema si:
    # 1. No es una stop word
    # 2. No es puntuación o símbolo
    # 3. Tiene más de 2 caracteres
    lemmas = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and len(token.text) > 2]
    
    return " ".join(lemmas)

# Aplicamos la función a tu columna de texto
print("Lematizando documentos... esto puede tardar un poco.")
df['text_cleaned'] = df['text'].astype(str).apply(lemmatize_text)

print("Muestra del texto lematizado:")
print(df['text_cleaned'].head(2))

✅ Modelo cargado correctamente!
Lematizando documentos... esto puede tardar un poco.
Muestra del texto lematizado:
0    child vaccine pneumococcal vaccine pcv ppsv pn...
1    child vaccine measle mump rubella mmr vaccine ...
Name: text_cleaned, dtype: str


In [13]:
print(df['text'])

0      Your Child's Vaccines: Pneumococcal Vaccines (...
1      Your Child's Vaccines: Measles, Mumps & Rubell...
2      Your Child's Vaccines: Influenza (Flu) Vaccine...
3      Your Child's Vaccines: Human Papillomavirus (H...
4      Your Child's Vaccines: Diphtheria, Tetanus & P...
                             ...                        
503    Scoliosis: Bracing\nOverview\nWhat Is a Scolio...
504    Scoliosis\nWhat Is Scoliosis?\nScoliosis is wh...
505    Scheuermann's Kyphosis\nWhat Is Scheuermann's ...
506    Rickets\nWhat Is Rickets?\nRickets is when a c...
507    Radial Dysplasia\nAlso Called: Radial Club Han...
Name: text, Length: 508, dtype: str


In [10]:
# Descargamos stop words
nltk.download('stopwords')
stop_words = stopwords.words('english') # O 'spanish' según tu dataset
# Añadimos palabras obvias que no ayudan a distinguir temas
stop_words.extend(['child', 'kid', 'kids', 'health', 'parent', 'help', 'children', 'may', 'use', 'get', 'doctor', 'doctors', 'people', 'baby', 'also', 'make', 'getting', 'like', 'type', 'teen', 'teens'])


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USUARIO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
# Usamos la columna ya lematizada
vectorizer = CountVectorizer(max_df=0.9, min_df=3, stop_words=stop_words)
data_vectorized = vectorizer.fit_transform(df['text_cleaned'])

# Entrenamos el modelo
lda = LatentDirichletAllocation(n_components=10, random_state=42)
lda.fit(data_vectorized)

# Función para ver los resultados limpios
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic #{topic_idx + 1}: "
        message += " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)

print_top_words(lda, vectorizer.get_feature_names_out(), 10)

Topic #1: vaccine skin rash cause infection disease month need give shot
Topic #2: infection antibiotic cause skin bacteria treat throat area sex prevent
Topic #3: brain seizure problem happen muscle cause epilepsy palsy cerebral medical
Topic #4: blood sugar level body test insulin care glucose diabetes need
Topic #5: ear thyroid syndrome hormone growth boy puberty girl problem cause
Topic #6: blood disease hepatitis symptom cell cause anemia gene liver mosquito
Topic #7: sleep eye night foot wear time day cause wake need
Topic #8: cell treatment tumor cancer problem bone cause therapy symptom test
Topic #9: infection symptom cause virus fever cough food pain spread hand
Topic #10: heart blood surgery problem body right ventricle cause need flow


In [ ]:
print("Preparando la visualización...")

panel = pyLDAvis.lda_model.prepare(lda, data_vectorized, vectorizer)

# Guardar
pyLDAvis.save_html(panel, 'analisis_temas.html')
print("✅ Archivo 'analisis_temas.html' generado con éxito.")

Preparando la visualización...
✅ Archivo 'analisis_temas.html' generado con éxito.
